## Step 1: Imports and reproducibility


In [ ]:
import sys, time, random
from pathlib import Path

root = Path.cwd().parent
sys.path.append(str(root))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import json

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import plot_model
from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import src.training.pretraining as Pretrain
from src.models.cnn import build_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

I0000 00:00:1792148831.824194  178867 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792148831.825635  178867 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792148832.153524  178867 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792148833.843988  178867 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

TensorFlow: 2.21.0
GPUs: []
TensorFlow: 2.21.0


E0000 00:00:1792148834.504700  178867 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Step 2: Load the preprocessed data


In [2]:
X_train, X_test, y_train, y_test, y_train_cat, y_test_cat, encoder, class_names = Pretrain.load()

X_train: (4916, 224, 224, 3) | y_train_cat: (4916, 4)
X_test : (2107, 224, 224, 3) | y_test_cat : (2107, 4)
Classes: [np.str_('glioma'), np.str_('meningioma'), np.str_('notumor'), np.str_('pituitary')]
Pixel range: 0.0 1.0


## Sanity check

In [3]:
Pretrain.sanityCheck(X_train, y_train_cat, y_train, y_test)

Train class counts: [1135 1151 1400 1230]
Test class counts : [486 494 600 527]
All checks passed


## Create a validation set

In [4]:
X_tr, X_val, y_tr, y_val, y_tr_int, y_val_int = train_test_split(
    X_train, y_train_cat, y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=SEED,
)

print("Train     :", X_tr.shape, y_tr.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test      :", X_test.shape, y_test_cat.shape)

Train     : (4178, 224, 224, 3) (4178, 4)
Validation: (738, 224, 224, 3) (738, 4)
Test      : (2107, 224, 224, 3) (2107, 4)


## class weights

In [5]:
weights = compute_class_weight("balanced", classes=np.unique(y_tr_int), y=y_tr_int)
class_weight = dict(enumerate(weights))

print("Class weights:", {k: round(v, 3) for k, v in class_weight.items()})

Class weights: {0: np.float64(1.082), 1: np.float64(1.068), 2: np.float64(0.878), 3: np.float64(1.0)}


## 5. Build and compile the model



In [6]:
model = build_model(len(class_names))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           51

 Total params: 2,425,124 (9.25 MB)

 Trainable params: 2,422,180 (9.24 MB)

 Non-trainable params: 2,944 (11.50 KB)

In [7]:
# Needs pydot and Graphviz installed
try:
    plot_model(model, to_file=str(root / "model.png"), show_shapes=True)
except Exception:
    print("plot_model skipped (install pydot and Graphviz to use it)")

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


## 6. Callbacks

- **ModelCheckpoint** saves the best model (lowest validation loss).
- **EarlyStopping** stops training when validation loss stops improving, and goes back to the best weights.

In [ ]:
models_dir = root / "models"
models_dir.mkdir(exist_ok=True)

callbacks = [
    ModelCheckpoint(str(models_dir / "best_model.h5"), monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]

## 7. Train (with timing)

In [ ]:
start = time.time()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
)

print(f"Training time: {(time.time() - start) / 60:.1f} min")

## 8. Learning curves

If training gets better while validation gets worse, the model is overfitting.

In [ ]:
plt.figure(figsize=(12, 4))

for i, metric in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, i + 1)
    plt.plot(history.history[metric], label="train")
    plt.plot(history.history["val_" + metric], label="validation")
    plt.title(metric)
    plt.xlabel("epoch")
    plt.legend()

plt.tight_layout()
plt.show()

## 9. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## 10. Classification report and confusion matrix

In [ ]:
probs = model.predict(X_test)
y_pred = np.argmax(probs, axis=1)

print(classification_report(y_test, y_pred, target_names=class_names))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## 11. Correct and incorrect predictions

In [ ]:


show(np.where(y_pred == y_test)[0], "Correct predictions")
show(np.where(y_pred != y_test)[0], "Incorrect predictions")

## 12. Save the model

In [ ]:
model.save(str(models_dir / "brain_tumor_cnn.h5"))
joblib.dump(class_names, models_dir / "class_names.pkl")   # needed by the API

print("Saved to", models_dir)